# Knowledge Graph Construction

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install sentence-transformers networkx openai numpy ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [4]:
EMBED_MODEL = "all-MiniLM-L6-v2"     
LLM_MODEL = "qwen2.5:7b"            

In [5]:
import json, os, re, numpy as np
import networkx as nx
from sentence_transformers import SentenceTransformer

In [6]:
try:
    from ollama import Client
    ollama_client = Client(host='http://localhost:11434')
    print("Using Ollama Client")
except ImportError:
    # Fallback to direct HTTP requests
    import requests
    ollama_client = None
    print("Using direct HTTP requests for Ollama")


SCRAPED_PATH = "scraped_content.txt"
VECTOR_DB_PATH = "chroma_db"
KG_JSON_PATH = "kg.json"

Using Ollama Client


In [7]:
from ollama import chat
import ollama
ollama.pull("qwen2.5:7b")
if ollama_client:
    response = ollama_client.chat(
        model="qwen2.5:7b",  
        messages=[
            {"role": "user", "content": "Hello world!"}
        ],
        stream = False
    )
    print("Model response:")
    print(response.message.content[:])
else:
    print("Ollama client not available, cannot run chat.")

Model response:
Hello! How can I assist you today?


Test KG Creation

In [121]:
from ollama import Client, pull

SCRAPED_PATH = "scraped_content.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"   # Local embedding model
LLM_MODEL = "qwen2.5:7b"           # Ollama local model
KG_FILE = "dummy_knowledge_graph.gpickle"
embedder = SentenceTransformer(EMBED_MODEL)

In [116]:
with open(SCRAPED_PATH, "r", encoding="utf-8") as f:
    full_text = f.read()

print("Loaded text:", len(full_text), "characters")

Loaded text: 12474141 characters


In [117]:
def extract_concepts(text):
    prompt = f"""
Extract ALL AI/ML concepts mentioned in this text.

For each concept include:
- name
- short definition (from context only)
- aliases / synonyms
- difficulty (easy/medium/hard)

Text:
{text[:20000]}  # limit for long text

Return JSON list with format:
[
  {{
    "name": "...",
    "definition": "...",
    "aliases": ["...", ...],
    "difficulty": "medium"
  }}
]
"""
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    data = json.loads(response.message.content)
    print("HERE I AM:", data)
    return data #["concepts"]

concepts = extract_concepts(full_text)
print("Extracted concepts:", len(concepts))


HERE I AM: [{'name': 'Introduction to MDP (Markov Decision Processes)', 'definition': 'A framework for modeling decision making in situations where outcomes are partly random and partly under the control of a decision maker. It consists of states, actions, rewards, transition probabilities, and policies.', 'aliases': ['MDP', 'Markov Decision Process'], 'difficulty': 'medium'}, {'name': 'Policy Iteration in MDP', 'definition': 'A method for finding the optimal policy by alternately improving the policy evaluation step (value iteration) and then using this value function to improve the policy.', 'aliases': ['Policy Improvement', 'Value Iteration'], 'difficulty': 'medium'}, {'name': 'Temporal Difference (TD) Learning', 'definition': 'A method for predicting state values or action-values by combining bootstrapping, which uses current estimates to improve future estimates, with eligibility traces, a mechanism that allows the updating of multiple states in one step.', 'aliases': ['TD(0)', 'S

In [118]:
# CREATE EMBEDDINGS FOR THE CONCEPTS
names = [c["name"] for c in concepts]
embeddings = embedder.encode(names, normalize_embeddings=True)


In [119]:
# BUILD KNOWLEDGE GRAPH
G = nx.DiGraph()

for concept, emb in zip(concepts, embeddings):
    G.add_node(
        concept["name"],
        type="concept",
        definition=concept["definition"],
        difficulty=concept["difficulty"],
        aliases=concept["aliases"],
        embedding=emb.astype(float).tolist()
    )

In [120]:
def cosine(a, b):
    return np.dot(a, b)

threshold = 0.55
for i, c1 in enumerate(concepts):
    for j, c2 in enumerate(concepts):
        if i >= j:
            continue
        sim = cosine(embeddings[i], embeddings[j])
        if sim >= threshold:
            G.add_edge(c1["name"], c2["name"], relation="related_to", weight=float(sim))
            G.add_edge(c2["name"], c1["name"], relation="related_to", weight=float(sim))

In [122]:
def infer_prereqs(concepts):
    concept_list = ", ".join([c["name"] for c in concepts])
    prompt = f"""
Here are some AI/ML concepts:

{concept_list}

Infer prerequisite relationships between concepts.
Use your understanding of AI—but ONLY return edges that are logically valid.

Return JSON:
{{
   "prerequisites": [
     {{"from": "probability", "to": "bayes rule"}},
     ...
   ]
}}
"""
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )

    print("HERE I AM1:",type(response),response)
    #return json.loads(response.message.content)["prerequisites"]
    return response


In [123]:
def parse_json_from_ollama(text):
    """
    Extract JSON content from Ollama response text.
    """
    # Search for a JSON code block first
    match = re.search(r"```json(.*?)```", text, re.DOTALL)
    if match:
        json_text = match.group(1).strip()
        return json.loads(json_text)
    
    # Fallback: try to parse the whole string
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("Warning: Could not parse JSON")
        return None

# Example usage:
raw_text = infer_prereqs(concepts).message.content
data = parse_json_from_ollama(raw_text)
prereq_edges = data.get("prerequisites", []) if data else []
print("Parsed prerequisites:", prereq_edges)

HERE I AM1: <class 'ollama._types.ChatResponse'> model='qwen2.5:7b' created_at='2025-12-06T00:58:42.3051245Z' done=True done_reason='stop' total_duration=4580873200 load_duration=69459500 prompt_eval_count=176 prompt_eval_duration=408906100 eval_count=204 eval_duration=3172803300 message=Message(role='assistant', content='```json\n{\n   "prerequisites": [\n     {"from": "MDP (Markov Decision Processes)", "to": "Probability Theory"},\n     {"from": "Policy Iteration in MDP", "to": "MDP (Markov Decision Processes)"},\n     {"from": "Temporal Difference (TD) Learning", "to": "Reinforcement Learning"},\n     {"from": "Reinforcement Learning", "to": "Markov Decision Processes (MDP)"},\n     {"from": "Maximum Likelihood Estimation (MLE)", "to": "Probability Theory"},\n     {"from": "Convolutional Neural Networks (CNNs)", "to": "Backpropagation"},\n     {"from": "Feature Extraction via Residual Networks (ResNet)", "to": "Convolutional Neural Networks (CNNs)"},\n     {"from": "Scene Understand

In [124]:
#prereq_edges = infer_prereqs(concepts)
for edge in prereq_edges:
    src = edge["from"]
    dst = edge["to"]
    if src in G.nodes() and dst in G.nodes():
        G.add_edge(src, dst, relation="prereq_of")


In [125]:
import pickle

with open(KG_FILE, "wb") as f:
    pickle.dump(G, f)
print(f"Graph saved → {KG_FILE}")
print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))


Graph saved → dummy_knowledge_graph.gpickle
Nodes: 14
Edges: 7


# Building the Knowledge Graph on the course website content

In [8]:
import os, json, re, numpy as np, pickle
import networkx as nx
from sentence_transformers import SentenceTransformer
from ollama import Client, pull


In [9]:
SCRAPED_PATH = "scraped_content.txt"
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "qwen2.5:7b"
KG_FILE = "knowledge_graph.pkl"  

CHUNK_SIZE = 60000 #20000
CHUNK_OVERLAP = 3000 #1000
EMBED_NORMALIZE = True
REL_SIM_THRESHOLD = 0.40


In [10]:
ollama_client = Client(host="http://localhost:11434")  
embedder = SentenceTransformer(EMBED_MODEL)


In [11]:
with open(SCRAPED_PATH, "r", encoding="utf-8") as f:
    full_text = f.read()
print("Loaded text:", len(full_text), "characters")

Loaded text: 12474141 characters


In [12]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks


In [13]:
def parse_json_from_ollama(text):

    results = []

    # 1. Extract ```json ... ``` blocks if present
    matches = re.findall(r"```json(.*?)```", text, re.DOTALL | re.IGNORECASE)
    chunks_to_parse = matches if matches else [text]

    for chunk in chunks_to_parse:
        # sanitize
        chunk = re.sub(r'[\x00-\x08\x0b-\x0c\x0e-\x1f]', '', chunk)
        chunk = chunk.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")

        # Attempt full JSON parse first
        try:
            parsed = json.loads(chunk)
            if isinstance(parsed, list):
                results.extend(parsed)
            else:
                results.append(parsed)
            continue
        except json.JSONDecodeError:
            pass

        # fallback: parse multiple JSON objects in sequence
        for obj_str in re.findall(r'\{.*?\}', chunk, re.DOTALL):
            try:
                results.append(json.loads(obj_str))
            except json.JSONDecodeError:
                continue

    return results



In [14]:
def extract_concepts(chunk):
    prompt = f"""
Extract ALL AI/ML concepts mentioned in this text.

For each concept include:
- name
- short definition (from context only)
- aliases / synonyms
- difficulty (easy/medium/hard)

Text:
{chunk}

Return JSON list with format:
[
  {{
    "name": "...",
    "definition": "...",
    "aliases": ["...", ...],
    "difficulty": "medium"
  }}
]
"""
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    return parse_json_from_ollama(response.message.content)


In [15]:
chunks = chunk_text(full_text)
all_concepts = []

for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)}")
    concepts_chunk = extract_concepts(chunk)
    if concepts_chunk:
        # flatten the list in case multiple JSON objects returned
        if isinstance(concepts_chunk, list):
            all_concepts.extend(concepts_chunk)
        else:
            all_concepts.append(concepts_chunk)


# Remove duplicates
unique_concepts = {c["name"]: c for c in all_concepts}
concepts = list(unique_concepts.values())
print("Total concepts extracted:", len(concepts))



Processing chunk 1/219
Processing chunk 2/219
Processing chunk 3/219
Processing chunk 4/219
Processing chunk 5/219
Processing chunk 6/219
Processing chunk 7/219
Processing chunk 8/219
Processing chunk 9/219
Processing chunk 10/219
Processing chunk 11/219
Processing chunk 12/219
Processing chunk 13/219
Processing chunk 14/219
Processing chunk 15/219
Processing chunk 16/219
Processing chunk 17/219
Processing chunk 18/219
Processing chunk 19/219
Processing chunk 20/219
Processing chunk 21/219
Processing chunk 22/219
Processing chunk 23/219
Processing chunk 24/219
Processing chunk 25/219
Processing chunk 26/219
Processing chunk 27/219
Processing chunk 28/219
Processing chunk 29/219
Processing chunk 30/219
Processing chunk 31/219
Processing chunk 32/219
Processing chunk 33/219
Processing chunk 34/219
Processing chunk 35/219
Processing chunk 36/219
Processing chunk 37/219
Processing chunk 38/219
Processing chunk 39/219
Processing chunk 40/219
Processing chunk 41/219
Processing chunk 42/219
P

In [20]:
# --- Append new concepts to the existing concepts list ---

new_concepts = [
    # Q1 Concepts
    {
        "name": "Dot-Product Attention",
        "definition": "A mechanism that computes attention weights by taking the dot product between a query vector and key vectors, scaling the result, applying softmax, and weighting the value vectors accordingly.",
        "aliases": ["scaled dot-product attention", "self-attention core"],
        "difficulty": "medium"
    },
    {
        "name": "Query, Key, and Value Vectors",
        "definition": "The three learned vector representations in attention mechanisms: queries represent what each token is seeking, keys represent what information each token contains, and values hold the actual token information to be aggregated.",
        "aliases": ["QKV", "attention vectors"],
        "difficulty": "easy"
    },
    {
        "name": "Self-Attention Mechanism",
        "definition": "A form of attention where the queries, keys, and values come from the same input sequence, allowing each token to dynamically weigh the importance of every other token.",
        "aliases": ["self-attention", "intra-attention"],
        "difficulty": "medium"
    },
    {
        "name": "Multi-Head Attention",
        "definition": "An attention mechanism that runs multiple self-attention operations in parallel, enabling the model to capture diverse relationships and patterns across the input sequence.",
        "aliases": ["MHA", "parallel attention heads"],
        "difficulty": "medium"
    },
    {
        "name": "Attention Score Scaling",
        "definition": "The practice of dividing the dot-product attention score by the square root of the key dimension to prevent extremely large values that destabilize the softmax function.",
        "aliases": ["scaled attention", "temperature scaling"],
        "difficulty": "easy"
    },

    # Q2 Concepts
    {
        "name": "Contrastive Learning",
        "definition": "A training paradigm where a model learns by pulling representations of matching pairs closer while pushing apart representations of mismatched pairs.",
        "aliases": ["contrastive loss", "InfoNCE"],
        "difficulty": "hard"
    },
    {
        "name": "Dual Encoder Architecture",
        "definition": "A model architecture where separate encoders are used to produce representations of different modalities, such as images and text, which are later compared using a similarity metric.",
        "aliases": ["separate encoders", "bi-encoder"],
        "difficulty": "medium"
    },
    {
        "name": "CLIP Text Encoder",
        "definition": "The Transformer-based network in CLIP responsible for converting natural language descriptions into dense embedding vectors.",
        "aliases": ["CLIP text embedder", "text transformer"],
        "difficulty": "medium"
    },
    {
        "name": "CLIP Image Encoder",
        "definition": "The convolutional or vision transformer network in CLIP used to convert images into embedding vectors for similarity comparison.",
        "aliases": ["CLIP image embedder", "image transformer"],
        "difficulty": "medium"
    },
    {
        "name": "Cosine Similarity in Embedding Space",
        "definition": "A similarity measure used to compare embedding vectors by computing the cosine of the angle between them. CLIP uses this to match images and text.",
        "aliases": ["embedding similarity", "cosine distance"],
        "difficulty": "easy"
    },
    {
        "name": "Zero-Shot Image Classification",
        "definition": "A classification method that requires no task-specific training; CLIP uses text prompts as class definitions and assigns the image to the class with the highest image–text similarity.",
        "aliases": ["prompt-based classification", "zero-shot learning"],
        "difficulty": "medium"
    },

    # Q3 Concepts
    {
        "name": "Jensen's Inequality",
        "definition": "A mathematical property stating that for a convex function, the function of an expectation is less than or equal to the expectation of the function. It is the core inequality used to derive variational bounds.",
        "aliases": ["convexity inequality"],
        "difficulty": "medium"
    },
    {
        "name": "Evidence Lower Bound (ELBO)",
        "definition": "A lower bound on the log marginal likelihood derived using Jensen's inequality. Maximizing the ELBO allows variational models like VAEs to approximate the true posterior distribution.",
        "aliases": ["variational lower bound", "ELBO"],
        "difficulty": "hard"
    },
    {
        "name": "Variational Inference",
        "definition": "A technique that approximates intractable posterior distributions with simpler distributions by optimizing a lower bound, typically the ELBO.",
        "aliases": ["VI", "variational methods"],
        "difficulty": "hard"
    },
    {
        "name": "Kullback–Leibler Divergence",
        "definition": "A measure of how one probability distribution diverges from another. In VAEs, KL divergence regularizes the latent distribution to match a prior.",
        "aliases": ["KL divergence", "relative entropy"],
        "difficulty": "medium"
    },
    {
        "name": "Variational Autoencoder Loss",
        "definition": "The objective function in VAEs consisting of a reconstruction loss and a KL divergence term, derived from maximizing the evidence lower bound (ELBO).",
        "aliases": ["VAE loss", "ELBO loss"],
        "difficulty": "medium"
    }
]

# Append them
try:
    concepts.extend(new_concepts)
except NameError:
    raise NameError("The variable 'concepts' is not defined. Ensure your concepts list exists before running this.")

print("Successfully added", len(new_concepts), "new concepts!")


Successfully added 16 new concepts!


In [21]:
concepts

[{'name': 'Camera Calibration Process',
  'definition': 'A series of steps to determine the intrinsic and extrinsic parameters of a camera, including its focal length, principal point, distortion coefficients, and pose relative to calibration views. This process is crucial for correcting lens distortions in images.',
  'aliases': ['camera parameter estimation', 'optical distortion correction'],
  'difficulty': 'medium'},
 {'name': 'RMS Reprojection Error',
  'definition': 'A metric used to evaluate the accuracy of a camera calibration. It measures the root mean square error between projected 3D points and their corresponding 2D points in the image plane after undistortion.',
  'aliases': ['root mean square error', 'reprojection error analysis'],
  'difficulty': 'medium'},
 {'name': 'Undistorted Image',
  'definition': 'An image where lens distortion has been corrected using a camera matrix and distortion coefficients, resulting in improved geometric accuracy of the captured scene.',
  

In [22]:
concept_names = []
for i in concepts:
    concept_names.append(i['name'])

concept_names



['Camera Calibration Process',
 'RMS Reprojection Error',
 'Undistorted Image',
 'Focal Length',
 'Principal Point',
 'Stream Data',
 'K',
 'A2',
 'Extract meaningful information from a complex text.',
 'UFO',
 'Extract JSON list from text',
 '6{(���n|�X�;Fr\x7f.���H��.�',
 'Xx3Ó',
 '6C',
 'Extract text from document',
 'Extract Information from Text',
 'Binary Data Sample',
 'Extract specific text from a complex document',
 'QE',
 'Code Parsing Task',
 'Example',
 'Bf)Xf',
 'LODAd',
 'Extract Text Between Tags',
 'Extract Text',
 'JSON Format',
 'Z4^h',
 'L�',
 'D&q���/�h�',
 'Gj~ߣQ��',
 '\u07b4',
 'Extract JSON list',
 'Example Entity',
 'stream',
 'endobj',
 'Extract Key Information from Text',
 'Z\x7f�',
 'Extract Information',
 'Z',
 'vcgt',
 'ndin',
 'mmod',
 'vcgp',
 'Extraction',
 'Text Puzzle',
 'IUn',
 'ɏIUn',
 'X',
 'C˙w',
 'h',
 'M.',
 'Extract Text from PDF',
 'Camera Calibration',
 'Extract JSON from Text',
 'extractedEntity',
 'Extract JSON list with format',
 'Extract T

In [23]:
import re

def is_valid_concept_name(name: str) -> bool:
    name = name.strip()

    if not name:
        return False
    
    # Remove corrupted unicode / gibberish (allow some East Asian chars)
    if re.search(r'[^\x00-\x7F]', name) and not re.search(r'[가-힣一-龥ぁ-んァ-ン]', name):
        return False

    # Remove too-short tokens
    if len(name) <= 3 and "AI" not in name:
        return False

    # Remove PDF operators
    pdf_ops = ["xref", "obj", "endobj", "stream", "endstream",
               "BitsPerComponent", "ColorSpace", "Subtype", "Filter"]
    if name in pdf_ops:
        return False

    # Remove font names / font-like tokens
    if re.search(r'(CM|LM|MathItalic|Regular|Bold|Font|CMBX)\d', name):
        return False
    if "Font" in name or "Regular" in name or "Bold" in name:
        return False

    # Remove programming tokens (all-lowercase single words)
    if re.match(r'^[a-z_]+$', name):
        return False

    # Remove random names / sample data
    blacklist = ["Gandalf", "Harry Potter", "Aldorin", "Example Name", "Zafer"]
    if name in blacklist:
        return False

    # Remove numeric-only or symbol-only
    if re.match(r'^[\d\W]+$', name):
        return False

    # Remove assignment/admin text
    admin_words = ["Assignment", "Repository", "Submission", "Document", "Preview"]
    if any(w in name for w in admin_words):
        return False

    # Remove patterns like subsubsection.3.6.4, equation.2.1
    if re.match(r'^[A-Za-z]+\.\d+(\.\d+)*$', name):
        return False

    return True

# Clean the concepts list
cleaned_concepts = [c for c in concepts if is_valid_concept_name(c['name'])]

print(f"Kept {len(cleaned_concepts)}/{len(concepts)} concepts")
print([c['name'] for c in cleaned_concepts[:20]]) 


Kept 220/309 concepts
['Camera Calibration Process', 'RMS Reprojection Error', 'Undistorted Image', 'Focal Length', 'Principal Point', 'Stream Data', 'Extract meaningful information from a complex text.', 'Extract JSON list from text', 'Extract text from document', 'Extract Information from Text', 'Binary Data Sample', 'Extract specific text from a complex document', 'Code Parsing Task', 'Example', 'Bf)Xf', 'LODAd', 'Extract Text Between Tags', 'Extract Text', 'JSON Format', 'Z4^h']


In [24]:
cleaned_concepts

[{'name': 'Camera Calibration Process',
  'definition': 'A series of steps to determine the intrinsic and extrinsic parameters of a camera, including its focal length, principal point, distortion coefficients, and pose relative to calibration views. This process is crucial for correcting lens distortions in images.',
  'aliases': ['camera parameter estimation', 'optical distortion correction'],
  'difficulty': 'medium'},
 {'name': 'RMS Reprojection Error',
  'definition': 'A metric used to evaluate the accuracy of a camera calibration. It measures the root mean square error between projected 3D points and their corresponding 2D points in the image plane after undistortion.',
  'aliases': ['root mean square error', 'reprojection error analysis'],
  'difficulty': 'medium'},
 {'name': 'Undistorted Image',
  'definition': 'An image where lens distortion has been corrected using a camera matrix and distortion coefficients, resulting in improved geometric accuracy of the captured scene.',
  

In [25]:
names = [c["name"] for c in cleaned_concepts]
embeddings = embedder.encode(names, normalize_embeddings=EMBED_NORMALIZE)

In [26]:
G = nx.DiGraph()
for concept, emb in zip(cleaned_concepts, embeddings):
    '''G.add_node(
        concept["name"],
        type="concept",
        definition=concept["definition"],
        difficulty=concept["difficulty"],
        aliases=concept["aliases"],
        embedding=emb.astype(float).tolist()
    )'''
    #for concept, emb in zip(concepts, embeddings):
    G.add_node(
        concept.get("name", "").strip(),
        type="concept",
        definition=concept.get("definition", "").strip(),
        difficulty=concept.get("difficulty", "unknown"),
        aliases=concept.get("aliases", []),
        embedding=emb.astype(float).tolist()
    )


In [27]:
def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)
    #return np.dot(a, b)

for i, c1 in enumerate(cleaned_concepts):
    for j, c2 in enumerate(cleaned_concepts):
        if i >= j:
            continue
        sim = cosine(embeddings[i], embeddings[j])
        if sim >= REL_SIM_THRESHOLD:
            G.add_edge(c1["name"], c2["name"], relation="related_to", weight=float(sim))
            G.add_edge(c2["name"], c1["name"], relation="related_to", weight=float(sim))


In [28]:
def infer_prereqs(cleaned_concepts):
    concept_list = ", ".join([c["name"] for c in cleaned_concepts])
    prompt = f"""
    Here are some AI/ML concepts:

    {concept_list}

    Infer prerequisite relationships between concepts.
    Return JSON with format:

    ```json
    {{
    "prerequisites": [
        {{"from": "concept A", "to": "concept B"}}
    ]
    }}
    """
    response = ollama_client.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        stream=False
    )
    data = parse_json_from_ollama(response.message.content)
    print("HERE:", data)

    if isinstance(data, dict):
        return data.get("prerequisites", [])

    elif isinstance(data, list):
        return data

    return []


prereq_edges = infer_prereqs(cleaned_concepts)

for edge in prereq_edges:

    # skip totally invalid formats
    if not isinstance(edge, dict):
        continue

    src = edge.get("from")
    dst = edge.get("to")

    # only source exists
    if src and not dst:
        if src not in G.nodes():
            G.add_node(src, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        continue

    # only destination exists
    if dst and not src:
        if dst not in G.nodes():
            G.add_node(dst, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        continue

    # both exist, add edge
    if src and dst:
        if src not in G.nodes():
            G.add_node(src, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        if dst not in G.nodes():
            G.add_node(dst, type="concept", definition="", difficulty="unknown",
                       aliases=[], embedding=[])
        G.add_edge(src, dst, relation="prereq_of")


HERE: [{'prerequisites': [{'from': 'Camera Calibration Process', 'to': 'RMS Reprojection Error'}, {'from': 'Camera Calibration Process', 'to': 'Undistorted Image'}, {'from': 'Camera Calibration Process', 'to': 'Focal Length'}, {'from': 'Camera Calibration Process', 'to': 'Principal Point'}, {'from': 'Camera Calibration', 'to': 'Camera Calibration Process'}, {'from': 'Extract meaningful information from a complex text.', 'to': 'Extract Information from Text'}, {'from': 'Extract meaningful information from a complex text.', 'to': 'Extract Key Information from Text'}, {'from': 'Extract JSON list from text', 'to': 'Extract Information from Text'}, {'from': 'Extract JSON list from text', 'to': 'Extract Key Information from Text'}, {'from': 'Extract specific text from a complex document', 'to': 'Extract Information from Text'}, {'from': 'Extract specific text from a complex document', 'to': 'Extract Key Information from Text'}, {'from': 'Code Parsing Task', 'to': 'Extract Information from Te

In [29]:
#with open(KG_FILE, "wb") as f:
with open("knowledge_graph1.pkl", "wb") as f:
    
    pickle.dump(G, f)

print(f"Graph saved → {"knowledge_graph1.pkl"}")
print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))


Graph saved → knowledge_graph1.pkl
Nodes: 219
Edges: 1831


In [30]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [31]:
import pandas as pd

df_nodes = pd.DataFrame([
    {
        "concept": n,
        "type": d.get("type"),
        "difficulty": d.get("difficulty"),
        "has_embedding": bool(d.get("embedding")),
        "aliases": ", ".join(d.get("aliases", [])),
        "definition": (d.get("definition")[:120] + "...") if d.get("definition") else ""
    }
    for n, d in G.nodes(data=True)
])

df_nodes.head(100)


,concept,type,difficulty,has_embedding,aliases,definition
0,Camera Calibration Process,concept,medium,True,"camera parameter estimation, optical distortio...",A series of steps to determine the intrinsic a...
1,RMS Reprojection Error,concept,medium,True,"root mean square error, reprojection error ana...",A metric used to evaluate the accuracy of a ca...
2,Undistorted Image,concept,easy,True,"corrected image, calibrated image",An image where lens distortion has been correc...
3,Focal Length,concept,easy,True,"focal distance, principal point",The distance between the optical center of the...
4,Principal Point,concept,easy,True,"image center, principal coordinates",The intersection of the optical axis and the i...
...,...,...,...,...,...,...
95,Extracting numerical data from a text,concept,easy,True,"numerical data extraction, number identification",The process of identifying and isolating numbe...
96,Parsing sequential data,concept,medium,True,"sequence parsing, sequential information extra...",The act of extracting specific sequences or pa...
97,Loss Value Trend,concept,medium,True,"Training Loss Over Epochs, Learning Curve",A pattern observed in the loss values over a s...
98,Loss Value Over Epochs,concept,medium,True,"Training Loss, Epoch-wise Loss Values",This dataset represents the loss values record...
